In [11]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_validate, KFold
from sklearn.metrics import make_scorer, mean_absolute_error, mean_squared_error, r2_score
from category_encoders import TargetEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

In [12]:
#Cleaning data 
data = pd.read_csv("2021-2022FootballPlayerStats.csv", sep=";", encoding="ISO-8859-1")
data = data.set_index("Player")

data["Age"] = data["Age"].fillna(data["Age"].mean())

#drop Born, Rk, Squad, Nation
columns_to_drop = ["Born", "Rk", "Squad", "Nation"]

data = data.drop(columns=columns_to_drop)

In [13]:
#Using RandomForests
name_player = "Erling Haaland"

player_with_goals = data.loc[[name_player]]

goals = data["Goals"]
data = data.drop("Goals", axis=1)

index_player = data.index.get_loc(name_player)

categorical_features = ["Pos", "Comp"]
numeric_features = [col for col in data.columns if col not in categorical_features]

data_train, data_test, goals_train, goals_test = train_test_split(
    data, goals, test_size=0.2, random_state=42
)

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", TargetEncoder(), categorical_features),
        ("num", StandardScaler(), numeric_features)
    ]
)

model = Pipeline([
    ("preprocess", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=400,
        max_depth=15,
        random_state=42
    ))
])

model.fit(data_train, goals_train)

player_without_goals = data.iloc[[index_player]]

print(player_with_goals["Goals"])
predicted_goals = model.predict(player_without_goals)
print(name_player + ": " + str(predicted_goals[0]))

Player
Erling Haaland    1.04
Name: Goals, dtype: float64
Erling Haaland: 0.9555250000000032


In [ ]:
#Error check
goals_pred = model.predict(data_test)

print(f"MAE:{mean_absolute_error(goals_test, goals_pred)}")
print(f"RMSE:{np.sqrt(mean_squared_error(goals_test, goals_pred))}")
print(f"R2:{r2_score(goals_test, goals_pred)}")

MAE:0.009985000144617421
RMSE:0.03183883988765376
R2:0.9736735080406903


In [ ]:
#Error check 
kf = KFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    'mae': make_scorer(mean_absolute_error),
    'rmse': make_scorer(lambda y_true, y_pred: np.sqrt(mean_squared_error(y_true, y_pred))),
    'r2': 'r2'
}

cv_results = cross_validate(model, data_train, goals_train, cv=kf, scoring=scoring)

print("MAE", cv_results['test_mae'])
print("mid MAE:", np.mean(cv_results['test_mae']))

print("RMSE:", cv_results['test_rmse'])
print("mid RMSE:", np.mean(cv_results['test_rmse']))

print("R²:", cv_results['test_r2'])
print("mid R²:", np.mean(cv_results['test_r2']))

MAE по фолдам: [0.01417123 0.0209799  0.01069706 0.01490155 0.01799605]
Среднее MAE: 0.015749156049171372
RMSE по фолдам: [0.03792668 0.1065325  0.03398971 0.05381871 0.13917659]
Среднее RMSE: 0.0742888380823449
R² по фолдам: [0.96445525 0.86674635 0.948958   0.92521702 0.81623563]
Среднее R²: 0.9043224482145437
